# 🎙️ InfiniteTalk (Wan2.1) Web 视频对口型工作室

基于 **Wan2.1 (14B) + InfiniteTalk + LightX2V 蒸馏加速**，针对 Google Colab 优化。

> 💡 **推荐硬件黄金甜蜜点**：**NVIDIA A100 (Standard 40GB)**（无需争抢 High-RAM 80GB，40GB 刚好全层常驻显存，单步仅需 13 秒，省点数近一半，秒级申领）；亦可在 **NVIDIA L4 (24GB)** 经济运行。
> 
> 📦 **代码仓库**：[https://github.com/ajaxsys/colab-lipsync](https://github.com/ajaxsys/colab-lipsync)

In [ ]:
# 1. 检查当前显卡规格
!nvidia-smi

In [ ]:
# 2. 克隆项目仓库并安装核心运行环境
import os

# 克隆主仓库
!git clone https://github.com/ajaxsys/colab-lipsync.git /content/colab-lipsync || (cd /content/colab-lipsync && git pull)
# 克隆 InfiniteTalk 依赖库
!git clone https://github.com/MeiGen-AI/InfiniteTalk.git /content/InfiniteTalk || true

# 安装依赖
%cd /content/colab-lipsync
!pip install -q -r requirements.txt

# 安装 Cloudflared 用于公网快速穿透
!if ! command -v cloudflared &> /dev/null; then wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb; fi
print("✅ 基础环境安装完成！")

In [ ]:
# 3. 准备模型权重 (直接从 Hugging Face 官方千兆 CDN 下载到本地高速 NVMe 固态硬盘，约 2~3 分钟)
# 注：本地 NVMe 固态硬盘读取速率达 2GB/s，比走 Google Drive FUSE 虚拟网络盘读取速度快 3~5 倍！
!python download_models.py

In [ ]:
# 4. 启动 Web 工作室与 Cloudflare 公网隧道
import subprocess, time, re

# 终止可能残留的旧进程
!pkill -9 -f "app.py" || true
!pkill -9 -f "cloudflared" || true

print("🚀 正在启动 Web 服务...")
web_proc = subprocess.Popen(["python3", "-u", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(3)
print("🌐 正在生成公网隧道链接...")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in cf_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "="*60)
if tunnel_url:
    print(f"🎉 Web 界面已成功上线！点击下方链接打开：\n👉 {tunnel_url}?__theme=light")
else:
    print("⚠️ 隧道链接生成中，请在终端查看日志")
print("="*60 + "\n")

---
### 💾 附：Google Drive 离线备份（可选兜底方案）
> 💡 **说明**：平时推荐优先使用「单元格 3」直接通过官方高速 CDN 下载（仅需 2~3 分钟即可写入本地高速 NVMe 固态硬盘，实测比走 Google Drive FUSE 网络虚拟盘更顺畅）。
>
> 此单元格为**纯环境兜底（Optional）**：若需防止境外网络抖动或希望离线持久化归档，可将 29GB 模型备份到 Google Drive 备用。

In [ ]:
# (可选) 挂载 Google Drive 并备份模型
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/colab-web-model
print("📦 开始同步模型文件到 Google Drive (约 29GB)... 请稍候")
!rsync -avP --delete /content/models/ /content/drive/MyDrive/colab-web-model/
print("🎉 备份完成！下次开启 Colab 可直接秒级挂载，无需重复下载。")